# Fase 3.8 — piloto AIGI, sem treino

Esta etapa testa se o [AIGI-Detection-Quality-Paradox](https://huggingface.co/datasets/Coxy7/AIGI-Detection-Quality-Paradox) pode fornecer pares úteis de fotografias reais e imagens geradas. A fonte publica 24 mil sintéticas de seis geradores, mas apenas metadados das reais; o piloto verifica a disponibilidade de uma amostra pareada. Nenhum checkpoint, divisão, avaliação externa ou treinamento é criado.

Execute as células 1–5 para preparar o runtime, depois a **célula 6** manualmente. A célula 7 resume o diagnóstico. CPU basta. O download de um shard Parquet completo (~495 MB) pode demorar; a célula mostra atividade. Não execute o treino da 3.7 novamente.


In [1]:
# 1. Dependências do piloto. CPU é suficiente.
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'cryptography', 'huggingface_hub', 'pyarrow', 'pillow'])
print('Dependências disponíveis.', flush=True)


Dependências disponíveis.


In [2]:
# 2. Drive e pastas novas; não altera experimentos antigos.
from google.colab import drive
from pathlib import Path
import shutil
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/TCC_Final')
for name in ('credenciais',):
    (ROOT / name).mkdir(parents=True, exist_ok=True)
print('Drive:', ROOT, '| livre reportado:', round(shutil.disk_usage(ROOT).free / 1e9, 2), 'GB')
print('/content livre:', round(shutil.disk_usage('/content').free / 1e9, 2), 'GB')

Mounted at /content/drive
Drive: /content/drive/MyDrive/TCC_Final | livre reportado: 5.13 GB
/content livre: 252.95 GB


In [3]:
# 3. Desbloqueio seguro da credencial GitHub existente (formatos antigo e novo).
import base64, getpass, json
from cryptography.fernet import Fernet, InvalidToken
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
credential_path = ROOT / 'credenciais/github.enc'
credential = json.loads(credential_path.read_text(encoding='utf-8'))
if not isinstance(credential, dict) or 'salt' not in credential or ('token' in credential) == ('ciphertext' in credential):
    raise ValueError('Formato da credencial GitHub não reconhecido')
if 'ciphertext' in credential:
    if credential.get('version') != 1: raise ValueError('Versão de credencial não suportada')
    iterations, encrypted = int(credential['iterations']), credential['ciphertext']
else:
    iterations, encrypted = 600000, credential['token']
if iterations < 600000: raise ValueError('KDF insuficiente')
password = getpass.getpass('Senha de desbloqueio do GitHub: ')
key = base64.urlsafe_b64encode(PBKDF2HMAC(algorithm=hashes.SHA256(), length=32, salt=base64.b64decode(credential['salt']), iterations=iterations).derive(password.encode()))
try:
    token = Fernet(key).decrypt(encrypted.encode()).decode()
except InvalidToken:
    raise ValueError('Senha incorreta ou credencial corrompida') from None
finally:
    del password, key
print('Credencial desbloqueada nesta sessão.')

Credencial desbloqueada nesta sessão.


In [4]:
# 4. Snapshot privado imutável da branch, somente leitura; sem token em URL ou Git.
import io, re, shutil, tempfile, time, urllib.request, urllib.error, zipfile
API = 'https://api.github.com/repos/vicemnovo/TCC_Final'
BRANCH = 'fase3-ajuste-fino'
def github_read(url):
    req = urllib.request.Request(url, headers={'Authorization': f'Bearer {token}', 'Accept': 'application/vnd.github+json', 'User-Agent': 'TCC-Fase38-Pilot-Colab'})
    try:
        with urllib.request.urlopen(req, timeout=120) as response:
            chunks, size, notice = [], 0, time.monotonic()
            while block := response.read(1024 * 1024):
                chunks.append(block); size += len(block)
                if time.monotonic() - notice >= 30:
                    print(f'Download do código: {size / 1048576:.1f} MiB', flush=True); notice = time.monotonic()
            return b''.join(chunks)
    except urllib.error.HTTPError as error:
        raise RuntimeError(f'GitHub HTTP {error.code}; confira token, repositório e branch') from None
try:
    commit = json.loads(github_read(f'{API}/branches/{BRANCH}'))['commit']['sha']
    assert re.fullmatch(r'[0-9a-f]{40}', commit)
    CODE_ROOT = Path('/content/tcc_codigo') / commit
    required = ('experimentos/fase3_8_aigi_piloto.py',)
    if not CODE_ROOT.exists():
        print('Baixando snapshot do GitHub...', flush=True)
        archive = github_read(f'{API}/zipball/{commit}')
        CODE_ROOT.parent.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(prefix='tcc38_', dir=CODE_ROOT.parent) as temporary:
            staged = Path(temporary)
            with zipfile.ZipFile(io.BytesIO(archive)) as bundle:
                for item in bundle.infolist():
                    parts = Path(item.filename).parts
                    if item.is_dir() or len(parts) < 2: continue
                    relative = Path(*parts[1:])
                    if relative.is_absolute() or '..' in relative.parts: raise ValueError('Snapshot contém caminho inseguro')
                    target = staged / relative
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with bundle.open(item) as source, target.open('wb') as output: shutil.copyfileobj(source, output)
            if not all((staged / name).is_file() for name in required):
                raise FileNotFoundError('Publique primeiro o piloto AIGI da Fase 3.8 na branch')
            staged.rename(CODE_ROOT)
    assert all((CODE_ROOT / name).is_file() for name in required)
finally:
    del token
print('Código sincronizado:', commit)

Baixando snapshot do GitHub...
Código sincronizado: 3c0ceff842c4f5bcb88ffcc6868ae6f43262d2a0


In [5]:
# 5. Comandos e progresso periódico; nenhum treino é iniciado aqui.
import os, queue, subprocess, sys, threading, time
DATA = CODE_ROOT / 'experimentos/fase3_8_aigi_piloto.py'
for path in (DATA,):
    if not path.is_file(): raise FileNotFoundError(path)
ENV = os.environ.copy()
ENV['TCC_CODE_ROOT'] = str(CODE_ROOT)
def run_live(label, command):
    print(label + ': iniciando', flush=True)
    process = subprocess.Popen(command, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = queue.Queue()
    def reader():
        for line in process.stdout: lines.put(line)
        lines.put(None)
    threading.Thread(target=reader, daemon=True).start()
    begin = last = time.monotonic()
    while True:
        try:
            line = lines.get(timeout=5)
            if line is None: break
            print(line, end='', flush=True)
            last = time.monotonic()
        except queue.Empty:
            if time.monotonic() - last >= 30:
                print(f'{label}: ativo há {(time.monotonic()-begin)/60:.1f} min; sem nova saída', flush=True)
                last = time.monotonic()
    code = process.wait()
    if code: raise RuntimeError(f'{label} falhou (código {code}); veja o log acima')
    print(f'{label}: concluído em {(time.monotonic()-begin)/60:.1f} min', flush=True)
print('Código:', CODE_ROOT)

Código: /content/tcc_codigo/3c0ceff842c4f5bcb88ffcc6868ae6f43262d2a0


In [6]:
# 6. PILOTO MANUAL: baixa CSV e um shard completos em /content; testa 16 fotos reais.
run_live('piloto AIGI 3.8', [sys.executable, '-u', str(DATA), 'pilot', '--samples', '16'])


piloto AIGI 3.8: iniciando
Baixando arquivo completo: real_images_metadata.csv | commit=9244882a51dbe33e658fe514488692155d20e5dd
Arquivo pronto: real_images_metadata.csv | 1.8 MB
Baixando arquivo completo: data/fake-00000-of-00015.parquet | commit=9244882a51dbe33e658fe514488692155d20e5dd
Arquivo pronto: data/fake-00000-of-00015.parquet | 495.2 MB
Metadados reais: {'SA-1B': 2972, 'Laion-2B-en-aesthetic': 856, 'MSCOCO': 133, 'CC3M': 39}
Shard sintético: 1600 | geradores: {'FLUX.1_dev': 1600} | formatos: {'JPEG': 1592, 'PNG': 8} | lado curto p10/med/p90: [1024, 1024, 1024]
UIDs pareáveis com URL HTTPS neste shard: 322
Foto real piloto: 1/16 | legível | lado curto=300
Foto real piloto: 2/16 | legível | lado curto=300
Foto real piloto: 3/16 | legível | lado curto=620
Foto real piloto: 4/16 | legível | lado curto=600
Foto real piloto: 5/16 | legível | lado curto=300
Foto real piloto: 6/16 | legível | lado curto=373
Foto real piloto: 7/16 | indisponível (HTTPError)
Foto real piloto: 8/16 | le

## Critério de avanço

O piloto deve mostrar UIDs pareáveis, imagens reais legíveis e resoluções adequadas, além de geradores e formatos do shard. Ele **não comprova** que a fonte inteira está disponível nem que a 3.8 terá melhor acurácia. Se poucos pares reais funcionarem, a coleta não deve prosseguir automaticamente; escolheremos outra fonte real antes de criar o experimento.


In [7]:
# 7. RESUMO SOMENTE-LEITURA: envie estes valores para decidir o preparo da 3.8.
import json
from pathlib import Path
summary = Path('/content/fase3_8_aigi_piloto/pilot_summary.json')
if not summary.is_file():
    raise FileNotFoundError('Execute primeiro a célula 6')
data = json.loads(summary.read_text(encoding='utf-8'))
for key in ('revision', 'real_sources', 'fake_in_shard', 'fake_generators',
            'fake_formats', 'fake_short_side_p10_med_p90',
            'paired_https_candidates_in_shard', 'real_probe_attempted',
            'real_probe_success', 'real_probe_short_side_512_plus',
            'real_probe_failures'):
    print(f'{key}: {data[key]}')


revision: 9244882a51dbe33e658fe514488692155d20e5dd
real_sources: {'SA-1B': 2972, 'Laion-2B-en-aesthetic': 856, 'MSCOCO': 133, 'CC3M': 39}
fake_in_shard: 1600
fake_generators: {'FLUX.1_dev': 1600}
fake_formats: {'JPEG': 1592, 'PNG': 8}
fake_short_side_p10_med_p90: [1024, 1024, 1024]
paired_https_candidates_in_shard: 322
real_probe_attempted: 16
real_probe_success: 15
real_probe_short_side_512_plus: 2
real_probe_failures: {'HTTPError': 1}
